# FlyRank — Colab execution
Latest local implementation. Run all cells in order on a CPU runtime. The setup downloads only the public starter dataset and verifies its hash. Package versions are recorded; results may differ from the local environment.


In [ ]:
from pathlib import Path
import os, sys, subprocess, urllib.request, hashlib
root = Path('/content/flyrank-evaluation')
root.mkdir(parents=True, exist_ok=True)
os.chdir(root)
for folder in ['scripts', 'data/raw', 'work/outputs', 'work/models']:
    (root/folder).mkdir(parents=True, exist_ok=True)
files = {'scripts/ml_utils.py': 'from __future__ import annotations\n\nimport json\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\n\nROOT = Path(__file__).resolve().parents[1]\nRAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"\nPROCESSED_DIR = ROOT / "data" / "processed"\nOUTPUT_DIR = ROOT / "outputs"\nCHART_DIR = OUTPUT_DIR / "charts"\n\n\nNUMERIC_COLUMNS = [\n    "search_volume",\n    "competition",\n    "cpc",\n    "word_count",\n    "char_count",\n    "impressions_90d",\n    "clicks_90d",\n    "pageviews_90d",\n    "sessions_90d",\n    "users_90d",\n    "engaged_sessions_90d",\n    "ai_sessions_90d",\n    "scroll_events_90d",\n    "days_with_impressions",\n    "days_with_sessions",\n    "impressions_last_30d",\n    "clicks_last_30d",\n    "sessions_last_30d",\n    "impressions_prev_30d",\n    "clicks_prev_30d",\n    "sessions_prev_30d",\n    "content_age_days",\n    "age_tier_order",\n    "days_since_last_update",\n    "ctr",\n    "avg_position",\n    "engagement_rate",\n    "scroll_rate",\n    "ai_traffic_pct",\n    "trend_pct",\n]\n\n\nBOOLEAN_COLUMNS: list[str] = []\n\n\nCATEGORICAL_COLUMNS = [\n    "competition_level",\n    "content_type",\n    "main_intent",\n    "provider_used",\n    "model_used",\n    "age_tier",\n    "freshness_tier",\n    "word_count_tier",\n    "char_count_tier",\n    "impression_tier",\n    "position_tier",\n    "trend_direction",\n]\n\n\nMODEL_NUMERIC_FEATURES = [\n    "search_volume",\n    "competition",\n    "cpc",\n    "word_count",\n    "char_count",\n    "log_impressions_90d",\n    "log_clicks_90d",\n    "log_sessions_90d",\n    "log_ai_sessions_90d",\n    "days_with_impressions",\n    "days_with_sessions",\n    "content_age_days",\n    "days_since_last_update",\n    "ctr",\n    "avg_position",\n    "engagement_rate",\n    "scroll_rate",\n    "ai_traffic_pct",\n]\n\n\nMODEL_CATEGORICAL_FEATURES = [\n    "competition_level",\n    "content_type",\n    "main_intent",\n    "age_tier",\n    "freshness_tier",\n    "word_count_tier",\n    "impression_tier",\n    "position_tier",\n]\n\n\ndef ensure_dirs() -> None:\n    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    CHART_DIR.mkdir(parents=True, exist_ok=True)\n\n\ndef write_json(path: Path, payload: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True))\n\n\ndef read_json(path: Path) -> dict:\n    return json.loads(path.read_text())\n\n\ndef display_path(path: Path | str) -> str:\n    resolved = Path(path).resolve()\n    try:\n        return str(resolved.relative_to(ROOT))\n    except ValueError:\n        return str(resolved)\n\n\ndef to_bool_series(series: pd.Series) -> pd.Series:\n    if series.dtype == bool:\n        return series.fillna(False)\n    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])\n\n\ndef normalize(series: pd.Series) -> pd.Series:\n    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)\n    minimum = values.min()\n    maximum = values.max()\n    if not np.isfinite(minimum) or not np.isfinite(maximum) or maximum == minimum:\n        return pd.Series(np.zeros(len(values)), index=values.index)\n    return (values - minimum) / (maximum - minimum)\n\n\ndef percentile_rank(series: pd.Series) -> pd.Series:\n    values = pd.to_numeric(series, errors="coerce").fillna(0)\n    return values.rank(method="average", pct=True).fillna(0)\n\n\ndef precision_at_k(y_true: Iterable[int], scores: Iterable[float], k: int) -> float:\n    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})\n    if frame.empty:\n        return 0.0\n    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))\n    return float(top["y"].mean()) if len(top) else 0.0\n\n\ndef safe_float(value: object, default: float = 0.0) -> float:\n    try:\n        number = float(value)\n        if np.isfinite(number):\n            return number\n    except Exception:\n        pass\n    return default\n\n\ndef simple_svg_bar_chart(\n    title: str,\n    labels: list[str],\n    values: list[float],\n    path: Path,\n    *,\n    width: int = 960,\n    height: int = 520,\n    color: str = "#6F4E7C",\n) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    labels = [str(label) for label in labels]\n    values = [safe_float(value) for value in values]\n    max_value = max(values) if values else 1\n    max_value = max(max_value, 1)\n    margin_left = 190\n    margin_right = 40\n    margin_top = 70\n    margin_bottom = 50\n    plot_width = width - margin_left - margin_right\n    plot_height = height - margin_top - margin_bottom\n    bar_gap = 10\n    bar_height = max(14, (plot_height - bar_gap * max(len(values) - 1, 0)) / max(len(values), 1))\n\n    lines = [\n        f\'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">\',\n        \'<rect width="100%" height="100%" fill="#ffffff"/>\',\n        f\'<text x="{width / 2}" y="34" text-anchor="middle" font-family="Arial" font-size="24" fill="#16232a">{escape_xml(title)}</text>\',\n    ]\n\n    for index, (label, value) in enumerate(zip(labels, values)):\n        y = margin_top + index * (bar_height + bar_gap)\n        bar_width = (value / max_value) * plot_width\n        lines.append(f\'<text x="{margin_left - 12}" y="{y + bar_height * 0.65:.1f}" text-anchor="end" font-family="Arial" font-size="13" fill="#27343b">{escape_xml(label[:32])}</text>\')\n        lines.append(f\'<rect x="{margin_left}" y="{y:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}" fill="{color}" rx="4"/>\')\n        lines.append(f\'<text x="{margin_left + bar_width + 8:.1f}" y="{y + bar_height * 0.65:.1f}" font-family="Arial" font-size="13" fill="#27343b">{value:,.3g}</text>\')\n\n    lines.append("</svg>")\n    path.write_text("\\n".join(lines))\n\n\ndef escape_xml(value: str) -> str:\n    return (\n        value.replace("&", "&amp;")\n        .replace("<", "&lt;")\n        .replace(">", "&gt;")\n        .replace(\'"\', "&quot;")\n        .replace("\'", "&apos;")\n    )\n', 'scripts/tune_capstone.py': '"""Bounded client-grouped model search; historical test split is scored only after selection."""\nimport hashlib,json,platform\nimport importlib.metadata\nimport joblib\nfrom sklearn.inspection import permutation_importance\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier,HistGradientBoostingClassifier\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.metrics import roc_auc_score,average_precision_score\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import OneHotEncoder\nfrom ml_utils import MODEL_NUMERIC_FEATURES,MODEL_CATEGORICAL_FEATURES\nROOT=Path(__file__).resolve().parents[1]\n\ndef measure(y,p):\n    order=np.argsort(-np.asarray(p),kind=\'stable\')\n    return {\'precision_at_20\':float(np.asarray(y)[order[:20]].mean()),\'precision_at_50\':float(np.asarray(y)[order[:50]].mean()),\'roc_auc\':float(roc_auc_score(y,p)),\'average_precision\':float(average_precision_score(y,p)),\'base_rate\':float(np.mean(y)),\'rows\':len(y)}\n\ndef features(d,expanded=False):\n    x=d.copy()\n    for c in [\'impressions_90d\',\'clicks_90d\',\'sessions_90d\',\'ai_sessions_90d\']:\n        x[\'log_\'+c]=np.log1p(x[c])\n    nums=list(MODEL_NUMERIC_FEATURES)\n    if expanded:\n        for a,b,name in [(\'impressions_90d\',\'days_with_impressions\',\'impressions_per_active_day\'),(\'sessions_90d\',\'days_with_sessions\',\'sessions_per_active_day\'),(\'days_since_last_update\',\'content_age_days\',\'relative_staleness\'),(\'char_count\',\'word_count\',\'chars_per_word\')]:\n            x[name]=x[a]/x[b].replace(0,np.nan);nums.append(name)\n        x[\'position_missing\']=x.avg_position.eq(0).astype(int);nums.append(\'position_missing\')\n        x.loc[x.avg_position.eq(0),\'avg_position\']=np.nan\n    x=x[nums+MODEL_CATEGORICAL_FEATURES].copy()\n    x[nums]=x[nums].replace([np.inf,-np.inf],np.nan)\n    for c in MODEL_CATEGORICAL_FEATURES:x[c]=x[c].fillna(\'unknown\').astype(str)\n    assert not any(c in x for c in [\'client_id\',\'content_id\',\'trend_pct\',\'trend_direction\',\'impressions_last_30d\',\'impressions_prev_30d\'])\n    return x,nums\n\ndef pipeline(model,nums):\n    return make_pipeline(ColumnTransformer([(\'num\',SimpleImputer(strategy=\'median\',add_indicator=True),nums),(\'cat\',OneHotEncoder(handle_unknown=\'ignore\',sparse_output=False),MODEL_CATEGORICAL_FEATURES)]),model)\n\ndef baseline(train,test):\n    # Fit percentile reference distributions and CTR medians using training rows only.\n    med=train.loc[train.avg_position.between(0,50,inclusive=\'right\')].groupby(\'position_tier\').ctr.median()\n    def gap(d):return (d.position_tier.map(med).fillna(d.ctr)-d.ctr).clip(lower=0)\n    def rank(ref,v):return np.searchsorted(np.sort(np.asarray(ref)),np.asarray(v),side=\'right\')/len(ref)\n    return .45*rank(train.impressions_90d,test.impressions_90d)+.35*rank(train.days_since_last_update,test.days_since_last_update)+.20*rank(gap(train),gap(test))*test.avg_position.gt(0).to_numpy()\n\ndef run():\n    d=pd.read_csv(ROOT/\'data/raw/content_refresh_anonymized.csv\')\n    d=d.loc[(d.impressions_90d>0)&(d.content_age_days>=90)].drop_duplicates(\'content_id\').reset_index(drop=True)\n    y=d.trend_direction.str.lower().eq(\'down\').astype(int).to_numpy()\n    clients=d.client_id.drop_duplicates().to_numpy(); hold=np.random.default_rng(42).permutation(clients)[:round(len(clients)*.2)]\n    test=d.client_id.isin(hold).to_numpy(); dev=np.flatnonzero(~test);testidx=np.flatnonzero(test)\n    assert set(d.loc[test,\'client_id\']).isdisjoint(set(d.loc[~test,\'client_id\']))\n    candidates={\n      \'forest_reference\':(False,RandomForestClassifier(n_estimators=200,max_depth=10,min_samples_leaf=25,class_weight=\'balanced_subsample\',n_jobs=4,random_state=42)),\n      \'forest_deeper\':(True,RandomForestClassifier(n_estimators=300,max_depth=18,min_samples_leaf=8,max_features=.8,class_weight=\'balanced_subsample\',n_jobs=4,random_state=42)),\n      \'extra_trees\':(True,ExtraTreesClassifier(n_estimators=300,max_depth=None,min_samples_leaf=8,max_features=.8,class_weight=\'balanced\',n_jobs=4,random_state=42)),\n      \'boosting_small\':(False,HistGradientBoostingClassifier(max_iter=200,learning_rate=.06,max_leaf_nodes=15,min_samples_leaf=40,l2_regularization=10,early_stopping=False,random_state=42)),\n      \'boosting_features\':(True,HistGradientBoostingClassifier(max_iter=200,learning_rate=.06,max_leaf_nodes=15,min_samples_leaf=40,l2_regularization=10,early_stopping=False,random_state=42)),\n      \'boosting_deeper\':(True,HistGradientBoostingClassifier(max_iter=250,learning_rate=.05,max_leaf_nodes=31,min_samples_leaf=60,l2_regularization=20,early_stopping=False,random_state=42))}\n    folds=list(GroupKFold(n_splits=4).split(dev,y[dev],d.loc[dev,\'client_id\']))\n    results={};fitted={}\n    for name,(expanded,model) in candidates.items():\n        x,nums=features(d,expanded);p=pipeline(model,nums);scores=[]\n        for fold,(a,b) in enumerate(folds):\n            tr,va=dev[a],dev[b]\n            assert set(d.loc[tr,\'client_id\']).isdisjoint(set(d.loc[va,\'client_id\']))\n            fitted_fold=clone(p).fit(x.iloc[tr],y[tr]);m=measure(y[va],fitted_fold.predict_proba(x.iloc[va])[:,1]);m[\'fold\']=fold+1;scores.append(m)\n        results[name]={\'folds\':scores,\'mean_precision_at_50\':float(np.mean([s[\'precision_at_50\'] for s in scores])),\'mean_precision_at_20\':float(np.mean([s[\'precision_at_20\'] for s in scores]))}\n        fitted[name]=(x,p)\n        print(name,round(results[name][\'mean_precision_at_50\'],3),flush=True)\n    # Selection uses only development cross-validation, never the historical holdout.\n    winner=max(results,key=lambda k:results[k][\'mean_precision_at_50\'])\n    test_results={\'rule_baseline\':measure(y[testidx],baseline(d.loc[~test],d.loc[test]))}\n    for name in dict.fromkeys([\'forest_reference\',winner]):\n        x,p=fitted[name];p.fit(x.iloc[dev],y[dev]);pred=p.predict_proba(x.iloc[testidx])[:,1];test_results[name]=measure(y[testidx],pred)\n        if name==winner:\n            (ROOT/\'work/models\').mkdir(exist_ok=True)\n            joblib.dump(p,ROOT/\'work/models/tuned_capstone.joblib\')\n            importance=permutation_importance(p,x.iloc[testidx],y[testidx],scoring=\'roc_auc\',n_repeats=3,random_state=42,n_jobs=1)\n            important=[{\'feature\':str(x.columns[i]),\'mean_auc_drop\':float(importance.importances_mean[i])} for i in np.argsort(-importance.importances_mean)[:5]]\n            top=np.argsort(-pred,kind=\'stable\')[:50];wrong=top[y[testidx][top]==0]\n            errors={\'false_positives_in_top_50\':int(len(wrong)),\'content_type_counts\':d.iloc[testidx[wrong]].content_type.value_counts().to_dict(),\'median_impressions_false_positive\':float(d.iloc[testidx[wrong]].impressions_90d.median()) if len(wrong) else None}\n    payload={\'selection\':\'Highest unweighted mean P@50 across four GroupKFold splits on the 26 development clients. Six fixed candidates; ties use declared order.\',\'selected_model\':winner,\'development_cv\':results,\'historical_holdout\':test_results,\'selected_error_summary\':errors,\'post_selection_permutation_importance\':important,\'packages\':{k:importlib.metadata.version(k) for k in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\']},\'script_sha256\':hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\'scope\':\'Exploratory current-state decline detection. Seed-42 holdout was used in earlier work and is not sealed. 90-day inputs overlap label window. No future or causal claim.\',\'dataset_sha256\':hashlib.sha256((ROOT/\'data/raw/content_refresh_anonymized.csv\').read_bytes()).hexdigest(),\'python\':platform.python_version(),\'model_parameters\':{k:v[1].get_params() for k,v in candidates.items()}}\n    out=ROOT/\'work/outputs/tuning_results.json\';out.write_text(json.dumps(payload,indent=2)+\'\\n\')\n    print(\'Selected:\',winner);print(json.dumps(test_results,indent=2))\n    lines=[\'# Model improvement experiment\',\'\',payload[\'scope\'],\'\',payload[\'selection\'],\'\',\'## Development validation\',\'\', \'| Model | Mean P@50 | Mean P@20 |\',\'|---|---:|---:|\']\n    lines += [f"| {k} | {v[\'mean_precision_at_50\']:.3f} | {v[\'mean_precision_at_20\']:.3f} |" for k,v in results.items()]\n    lines += [\'\',\'Fold base rates: \'+\', \'.join(f"{r[\'base_rate\']:.3f}" for r in results[\'forest_reference\'][\'folds\'])+\'. Individual fold metrics are in the results JSON.\']\n    lines += [\'\',f\'Selected before holdout scoring: **{winner}**.\',\'\',\'## Historical holdout comparison\',\'\',\'| Method | P@50 | P@20 | ROC-AUC | Base rate |\',\'|---|---:|---:|---:|---:|\']\n    lines += [f"| {k} | {v[\'precision_at_50\']:.3f} | {v[\'precision_at_20\']:.3f} | {v[\'roc_auc\']:.3f} | {v[\'base_rate\']:.3f} |" for k,v in test_results.items()]\n    lines += [\'\',\'The reference forest here uses training-only median imputation, missingness flags and category encoding. It is a controlled comparator, not an exact replay of the previous zero-imputed 68% run. Stable ranking resolves ties by dataset order.\',\'\',f"Selected model has {errors[\'false_positives_in_top_50\']} false positives among its top 50.",\'\',\'Feature additions: activity per active day, relative staleness, characters per word and missing-position flag. No trend labels, comparison-window ingredients or IDs enter the model. Preprocessors fit within each training fold. The rule baseline also fits its distributions and tier medians on training clients only.\',\'\',\'Do not replace the published capstone headline merely because this historical split improves. Inspect the development fold results and validate on genuinely new clients and non-overlapping future outcomes before deployment.\',\'\',\'Reproduce with the existing requirements-capstone.lock.txt environment and run work/notebooks/model_improvement.ipynb top to bottom. Aggregate evidence is in work/outputs/tuning_results.json.\']\n    lines += [\'\',\'## Model inspection\',\'\', \'Top features by post-selection permutation ROC-AUC drop: \'+\', \'.join(f"{r[\'feature\']} ({r[\'mean_auc_drop\']:.4f})" for r in important)+\'. This diagnostic is not used for selection and does not establish causality.\',\'\', \'Selected boosting_small uses the original feature list plus imputation missingness flags; the extra activity ratios did not improve validation performance.\',\'\', \'Trained pipeline saved locally to work/models/tuned_capstone.joblib (excluded from Git). Feed it the feature frame returned by features(data, expanded=False); it includes fitted imputation and encoding. Only load model binaries from trusted sources.\']\n    (ROOT/\'work/model_improvement.md\').write_text(\'\\n\'.join(lines)+\'\\n\')\n    return payload\n\nif __name__==\'__main__\':run()\n', 'scripts/render_capstone_report.py': '"""Render all capstone narratives from the notebook\'s aggregate JSON."""\nimport html\nimport json\nimport re\nfrom pathlib import Path\n\n\ndef render(root):\n    m = json.loads((root / \'work/outputs/capstone_model_metrics.json\').read_text())\n    rf, base = m[\'random_forest\'], m[\'baseline\']\n    summary = (f"Random Forest Precision@50 is {rf[\'precision_at_50\']:.0%}, versus "\n               f"{base[\'precision_at_50\']:.0%} for the baseline "\n               f"({100*(rf[\'precision_at_50\']-base[\'precision_at_50\']):.0f} percentage points higher). "\n               f"The seed-42 holdout contains {m[\'test_rows\']:,} pages from {m[\'n_holdout_clients\']} clients, "\n               f"with a declining-label base rate of {m[\'test_base_rate\']:.1%}.")\n    md = [\'# Content refresh prioritisation\', \'\', \'Prajwal Ravi Devadiga · FlyRank ML Internship\', \'\']\n    body = [\'<h1>Content refresh prioritisation</h1><p class="author">Prajwal Ravi Devadiga · FlyRank ML Internship</p>\']\n\n    def section(title, paragraphs):\n        md.extend([\'## \' + title, \'\'])\n        body.append(\'<h2>\' + html.escape(title) + \'</h2>\')\n        for p in paragraphs:\n            md.extend([p, \'\'])\n            body.append(\'<p>\' + html.escape(p) + \'</p>\')\n\n    def table(headers, rows):\n        md.extend([\'| \' + \' | \'.join(headers) + \' |\', \'| \' + \' | \'.join([\'---\']*len(headers)) + \' |\'])\n        md.extend(\'| \' + \' | \'.join(map(str, row)) + \' |\' for row in rows)\n        md.append(\'\')\n        body.append(\'<div style="overflow-x:auto"><table><thead><tr>\' + \'\'.join(\'<th scope="col">\'+html.escape(h)+\'</th>\' for h in headers) + \'</tr></thead><tbody>\')\n        for row in rows:\n            body.append(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(v))+\'</td>\' for v in row)+\'</tr>\')\n        body.append(\'</tbody></table></div>\')\n\n    section(\'Measured result\', [summary, \'This exploratory study ranks pages by observed impression decline. It does not predict future traffic or demonstrate that refreshing a page improves its performance.\'])\n    section(\'Question and data\', [\n        \'Can content and search signals help an editor prioritise a hypothetical budget of 50 page reviews? Precision@50 measures how many selected pages carry the observed declining label.\',\n        f"The bundled FlyRank sample contains {sum(m[\'label_distribution\'].values()):,} retained pages across {m[\'n_clients_total\']} pseudonymous clients. Keep pages with positive 90-day impressions and content age of at least 90 days; deduplicate content IDs. The target is trend_direction == down.",\n        \'Direct trend variables, last/previous 30-day ingredients and IDs are excluded from features. IDs are used only for deduplication and grouped splitting. Published outputs are aggregate summaries; no action queue or additional dataset is committed.\'\n    ])\n    section(\'Baseline and models\', [\n        \'Baseline score = 0.45 × visibility percentile + 0.35 × staleness percentile + 0.20 × positive CTR-gap percentile. Expected CTR is the median for each position tier among pages with positions above zero and at most 50; unmapped tiers fall back to observed CTR. Nonpositive positions receive zero CTR-gap contribution.\',\n        \'Baseline ranks, CTR medians and categorical vocabulary are computed from the full unlabelled snapshot. This transductive setup is retained to reproduce the original experiment. Model fitting uses training clients only.\',\n        \'The Random Forest uses 200 trees, maximum depth 10, minimum leaf size 25, balanced_subsample weights and random_state=42. Comparators are scaled balanced Logistic Regression (max_iter=1000) and a balanced depth-5 Decision Tree (minimum leaf size 50). Inputs are the 18 numeric and 8 categorical features listed in scripts/ml_utils.py.\'\n    ])\n    section(\'Client-grouped holdout\', [\n        f"Seed 42 selects {m[\'n_holdout_clients\']} of {m[\'n_clients_total\']} clients for testing, leaving {m[\'train_rows\']:,} training pages. Assertions verify no client overlap and exclude forbidden features. Models share the same holdout.",\n        \'These models have been compared using this holdout, so it is development evidence rather than an untouched final test. The table reports observed point estimates, without confidence intervals.\'\n    ])\n    table([\'Method\',\'ROC-AUC\',\'Precision@20\',\'Precision@50\'], [[name.replace(\'_\',\' \').title(),*[f\'{m[name][k]:.3f}\' for k in [\'roc_auc\',\'precision_at_20\',\'precision_at_50\']]] for name in [\'baseline\',\'logistic_regression\',\'decision_tree\',\'random_forest\']])\n    section(\'Actually executed split sensitivity\', [\'For each split seed, refit the Random Forest with its model seed fixed at 42. Recompute metrics against that split’s baseline and base rate. These three chosen splits are a sensitivity check, not a confidence interval or evidence that every unseen client will benefit.\'])\n    table([\'Split seed\',\'Test pages\',\'Base rate\',\'RF P@50\',\'Baseline P@50\'], [[r[\'split_seed\'],r[\'test_rows\'],*[f\'{r[k]:.3f}\' for k in [\'test_base_rate\',\'rf_precision_at_50\',\'baseline_precision_at_50\']]] for r in m[\'seed_sensitivity\']])\n    section(\'Interpretation\', [f"In the canonical holdout, {round(50*rf[\'precision_at_50\'])} of the top 50 pages carry the declining label; {50-round(50*rf[\'precision_at_50\'])} do not. This is a measured label count, not a forecast of time saved or traffic recovered.", \'The following impurity-based feature importances describe the fitted forest. Correlated features and high-cardinality inputs can affect these values; they do not establish causes or the direction of effects.\'])\n    table([\'Feature\',\'Importance\'], [[r[\'feature\'],f"{r[\'importance\']:.4f}"] for r in m[\'feature_importance\']])\n    section(\'Practical use and limitations\', [\n        \'The local demonstration queue includes training pages, and its action rules use the observed trend label. It is not validation evidence. Review the trend in the source analytics before deciding whether any content needs work.\',\n        \'The 90-day aggregates overlap the window used to define decline. Client grouping does not remove this temporal overlap. A future-prediction claim requires non-overlapping outcome windows.\',\n        \'Only six clients are held out in the canonical split; client mix can materially change results. Full-snapshot preprocessing also limits claims about completely unseen data.\',\n        \'Next experiment: fit preprocessing only on training clients, evaluate on future non-overlapping outcomes and reserve a new untouched set after model selection. Earlier weekly notebooks and supplied starter outputs are historical experiments, not the canonical capstone results.\'\n    ])\n    section(\'Reproduce and audit\', [f"Executed with Python {m[\'provenance\'][\'python\']}. Exact package versions are recorded in requirements-capstone.lock.txt; input and code hashes are in work/outputs/capstone_model_metrics.json. The notebook generates this page and work/capstone_report.md from that same JSON. Numerical results can change with library versions."])\n    command=\'python3.14 -m venv .venv\\n.venv/bin/pip install -r requirements-capstone.lock.txt\\n.venv/bin/python scripts/run_capstone.py\'\n    md.extend([\'```bash\',command,\'```\',\'\'])\n    body.append(\'<pre>\'+html.escape(command)+\'</pre>\')\n    section(\'Acknowledgments and data credit\', [\'Dataset and starter foundation courtesy of FlyRank. Original starter scripts and teaching materials remain credited to their contributors.\'])\n    md.extend([\'[FlyRank](https://flyrank.ai) · [Executed notebook](notebooks/capstone.ipynb) · [Aggregate metrics](outputs/capstone_model_metrics.json)\',\'\'])\n    body.append(\'<p><a href="https://flyrank.ai">FlyRank</a> · <a href="https://github.com/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb">Executed notebook</a> · <a href="https://github.com/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/outputs/capstone_model_metrics.json">Aggregate metrics</a></p>\')\n    (root/\'work/capstone_report.md\').write_text(\'\\n\'.join(md))\n    page=root/\'index.html\'\n    head=page.read_text().split(\'<body>\')[0]\n    page.write_text(head+\'<body>\\n<div class="container">\\n\'+\'\\n\'.join(body)+\'\\n</div>\\n</body>\\n</html>\\n\')\n    readme=root/\'README.md\'\n    status=\'**Evaluation status:** \'+summary+\' Regenerate the notebook, metrics and reports with `python scripts/run_capstone.py` using the pinned capstone environment. These are exploratory current-state results with overlapping windows and full-snapshot preprocessing; see the report for limitations.\'\n    readme.write_text(re.sub(r\'^\\*\\*Evaluation status:\\*\\*.*$\',lambda _:status,readme.read_text(),flags=re.M))\n\n\nif __name__ == \'__main__\':\n    render(Path(__file__).resolve().parents[1])\n', 'index.html': '<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>Content Refresh Prioritization — FlyRank ML Internship 2026</title>\n    <style>\n        :root {\n            --brand-dark: #16232A;\n            --brand-text: #27343B;\n            --brand-muted: #65737A;\n            --brand-purple: #6F4E7C;\n            --brand-teal: #426B69;\n            --brand-blue: #4E79A7;\n            --bg-light: #F6F4F7;\n            --line: #D9DEE2;\n        }\n        * { box-sizing: border-box; margin: 0; padding: 0; }\n        body {\n            font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif;\n            color: var(--brand-text);\n            line-height: 1.7;\n            background: #fff;\n        }\n        .container { max-width: 880px; margin: 0 auto; padding: 2rem 1.5rem; }\n        h1 { color: var(--brand-dark); font-size: 2rem; margin-bottom: 0.3rem; }\n        h2 { color: var(--brand-dark); font-size: 1.4rem; margin: 2.5rem 0 0.8rem; border-bottom: 2px solid var(--line); padding-bottom: 0.3rem; }\n        h3 { color: var(--brand-purple); font-size: 1.1rem; margin: 1.5rem 0 0.5rem; }\n        p { margin-bottom: 1rem; }\n        .subtitle { color: var(--brand-muted); font-size: 1rem; margin-bottom: 2rem; }\n        .author { color: var(--brand-muted); font-size: 0.9rem; margin-bottom: 0.5rem; }\n        .abstract {\n            background: var(--bg-light);\n            border-left: 4px solid var(--brand-purple);\n            padding: 1.2rem 1.5rem;\n            margin: 1.5rem 0;\n            border-radius: 0 8px 8px 0;\n        }\n        table { width: 100%; border-collapse: collapse; margin: 1rem 0 1.5rem; font-size: 0.92rem; }\n        th { background: var(--brand-dark); color: #fff; padding: 0.6rem 0.8rem; text-align: left; }\n        td { padding: 0.5rem 0.8rem; border-bottom: 1px solid var(--line); }\n        tr:nth-child(even) { background: var(--bg-light); }\n        .metric-cards {\n            display: grid;\n            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));\n            gap: 1rem;\n            margin: 1.5rem 0;\n        }\n        .metric-card {\n            background: #fff;\n            border: 1px solid var(--line);\n            border-radius: 8px;\n            padding: 1.2rem;\n            text-align: center;\n        }\n        .metric-card .value { font-size: 1.8rem; font-weight: 700; color: var(--brand-dark); }\n        .metric-card .label { font-size: 0.8rem; color: var(--brand-muted); margin-top: 0.3rem; }\n        code {\n            background: var(--bg-light);\n            padding: 0.15rem 0.4rem;\n            border-radius: 3px;\n            font-size: 0.88em;\n            font-family: \'SF Mono\', \'Fira Code\', monospace;\n        }\n        pre {\n            background: var(--brand-dark);\n            color: #e0e0e0;\n            padding: 1rem 1.2rem;\n            border-radius: 8px;\n            overflow-x: auto;\n            font-size: 0.88rem;\n            margin: 1rem 0;\n        }\n        ul, ol { margin: 0.5rem 0 1rem 1.5rem; }\n        li { margin-bottom: 0.3rem; }\n        .highlight { color: var(--brand-purple); font-weight: 600; }\n        .warning {\n            background: #FFF8E1;\n            border-left: 4px solid #F28E2B;\n            padding: 0.8rem 1.2rem;\n            margin: 1rem 0;\n            border-radius: 0 8px 8px 0;\n            font-size: 0.92rem;\n        }\n        footer {\n            margin-top: 3rem;\n            padding: 1.5rem 0;\n            border-top: 2px solid var(--line);\n            text-align: center;\n            color: var(--brand-muted);\n            font-size: 0.85rem;\n        }\n        footer a { color: var(--brand-purple); }\n        a { color: var(--brand-blue); text-decoration: none; }\n        a:hover { text-decoration: underline; }\n    </style>\n</head>\n<body>\n<div class="container">\n<h1>Content refresh prioritisation</h1><p class="author">Prajwal Ravi Devadiga · FlyRank ML Internship</p>\n<h2>Measured result</h2>\n<p>Random Forest Precision@50 is 68%, versus 42% for the baseline (26 percentage points higher). The seed-42 holdout contains 2,325 pages from 6 clients, with a declining-label base rate of 39.1%.</p>\n<p>This exploratory study ranks pages by observed impression decline. It does not predict future traffic or demonstrate that refreshing a page improves its performance.</p>\n<h2>Question and data</h2>\n<p>Can content and search signals help an editor prioritise a hypothetical budget of 50 page reviews? Precision@50 measures how many selected pages carry the observed declining label.</p>\n<p>The bundled FlyRank sample contains 30,000 retained pages across 32 pseudonymous clients. Keep pages with positive 90-day impressions and content age of at least 90 days; deduplicate content IDs. The target is trend_direction == down.</p>\n<p>Direct trend variables, last/previous 30-day ingredients and IDs are excluded from features. IDs are used only for deduplication and grouped splitting. Published outputs are aggregate summaries; no action queue or additional dataset is committed.</p>\n<h2>Baseline and models</h2>\n<p>Baseline score = 0.45 × visibility percentile + 0.35 × staleness percentile + 0.20 × positive CTR-gap percentile. Expected CTR is the median for each position tier among pages with positions above zero and at most 50; unmapped tiers fall back to observed CTR. Nonpositive positions receive zero CTR-gap contribution.</p>\n<p>Baseline ranks, CTR medians and categorical vocabulary are computed from the full unlabelled snapshot. This transductive setup is retained to reproduce the original experiment. Model fitting uses training clients only.</p>\n<p>The Random Forest uses 200 trees, maximum depth 10, minimum leaf size 25, balanced_subsample weights and random_state=42. Comparators are scaled balanced Logistic Regression (max_iter=1000) and a balanced depth-5 Decision Tree (minimum leaf size 50). Inputs are the 18 numeric and 8 categorical features listed in scripts/ml_utils.py.</p>\n<h2>Client-grouped holdout</h2>\n<p>Seed 42 selects 6 of 32 clients for testing, leaving 27,675 training pages. Assertions verify no client overlap and exclude forbidden features. Models share the same holdout.</p>\n<p>These models have been compared using this holdout, so it is development evidence rather than an untouched final test. The table reports observed point estimates, without confidence intervals.</p>\n<div style="overflow-x:auto"><table><thead><tr><th scope="col">Method</th><th scope="col">ROC-AUC</th><th scope="col">Precision@20</th><th scope="col">Precision@50</th></tr></thead><tbody>\n<tr><td>Baseline</td><td>0.646</td><td>0.400</td><td>0.420</td></tr>\n<tr><td>Logistic Regression</td><td>0.700</td><td>0.350</td><td>0.400</td></tr>\n<tr><td>Decision Tree</td><td>0.742</td><td>0.650</td><td>0.660</td></tr>\n<tr><td>Random Forest</td><td>0.747</td><td>0.700</td><td>0.680</td></tr>\n</tbody></table></div>\n<h2>Actually executed split sensitivity</h2>\n<p>For each split seed, refit the Random Forest with its model seed fixed at 42. Recompute metrics against that split’s baseline and base rate. These three chosen splits are a sensitivity check, not a confidence interval or evidence that every unseen client will benefit.</p>\n<div style="overflow-x:auto"><table><thead><tr><th scope="col">Split seed</th><th scope="col">Test pages</th><th scope="col">Base rate</th><th scope="col">RF P@50</th><th scope="col">Baseline P@50</th></tr></thead><tbody>\n<tr><td>42</td><td>2325</td><td>0.391</td><td>0.680</td><td>0.420</td></tr>\n<tr><td>7</td><td>4280</td><td>0.611</td><td>0.840</td><td>0.760</td></tr>\n<tr><td>123</td><td>3090</td><td>0.549</td><td>0.980</td><td>0.980</td></tr>\n</tbody></table></div>\n<h2>Interpretation</h2>\n<p>In the canonical holdout, 34 of the top 50 pages carry the declining label; 16 do not. This is a measured label count, not a forecast of time saved or traffic recovered.</p>\n<p>The following impurity-based feature importances describe the fitted forest. Correlated features and high-cardinality inputs can affect these values; they do not establish causes or the direction of effects.</p>\n<div style="overflow-x:auto"><table><thead><tr><th scope="col">Feature</th><th scope="col">Importance</th></tr></thead><tbody>\n<tr><td>log_impressions_90d</td><td>0.1318</td></tr>\n<tr><td>days_with_impressions</td><td>0.1305</td></tr>\n<tr><td>avg_position</td><td>0.1108</td></tr>\n<tr><td>content_age_days</td><td>0.0908</td></tr>\n<tr><td>age_tier_365+</td><td>0.0376</td></tr>\n<tr><td>word_count</td><td>0.0370</td></tr>\n<tr><td>log_clicks_90d</td><td>0.0366</td></tr>\n<tr><td>char_count</td><td>0.0362</td></tr>\n<tr><td>days_with_sessions</td><td>0.0348</td></tr>\n<tr><td>ctr</td><td>0.0342</td></tr>\n</tbody></table></div>\n<h2>Practical use and limitations</h2>\n<p>The local demonstration queue includes training pages, and its action rules use the observed trend label. It is not validation evidence. Review the trend in the source analytics before deciding whether any content needs work.</p>\n<p>The 90-day aggregates overlap the window used to define decline. Client grouping does not remove this temporal overlap. A future-prediction claim requires non-overlapping outcome windows.</p>\n<p>Only six clients are held out in the canonical split; client mix can materially change results. Full-snapshot preprocessing also limits claims about completely unseen data.</p>\n<p>Next experiment: fit preprocessing only on training clients, evaluate on future non-overlapping outcomes and reserve a new untouched set after model selection. Earlier weekly notebooks and supplied starter outputs are historical experiments, not the canonical capstone results.</p>\n<h2>Reproduce and audit</h2>\n<p>Executed with Python 3.14.0. Exact package versions are recorded in requirements-capstone.lock.txt; input and code hashes are in work/outputs/capstone_model_metrics.json. The notebook generates this page and work/capstone_report.md from that same JSON. Numerical results can change with library versions.</p>\n<pre>python3.14 -m venv .venv\n.venv/bin/pip install -r requirements-capstone.lock.txt\n.venv/bin/python scripts/run_capstone.py</pre>\n<h2>Acknowledgments and data credit</h2>\n<p>Dataset and starter foundation courtesy of FlyRank. Original starter scripts and teaching materials remain credited to their contributors.</p>\n<p><a href="https://flyrank.ai">FlyRank</a> · <a href="https://github.com/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb">Executed notebook</a> · <a href="https://github.com/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/outputs/capstone_model_metrics.json">Aggregate metrics</a></p>\n</div>\n</body>\n</html>\n', 'README.md': '# FlyRank · Content Refresh Prioritisation\n\n**Applied ML internship workspace · Prajwal Ravi Devadiga**\n\nA capstone exploring how search-performance signals can help content editors decide which pages to review first. The workflow covers problem framing, feature and leakage audits, a transparent rule baseline, model comparison, client-grouped validation and an action playbook.\n\n[Capstone report](work/capstone_report.md) · [Capstone notebook](work/notebooks/capstone.ipynb) · [Validation audit](work/notebooks/w06_validation_audit.ipynb) · [Action playbook](work/notebooks/w07_action_playbook.ipynb)\n\n## Start with the evidence\n\n| What to review | Where to look |\n|---|---|\n| Problem, modelling choices and limitations | [Capstone report](work/capstone_report.md) |\n| Experiment implementation | [Assignment notebooks](work/notebooks/) and [capstone notebook](work/notebooks/capstone.ipynb) |\n| Saved evaluation summaries | [Metrics JSON](work/outputs/capstone_model_metrics.json) (includes model comparison, seed sensitivity and feature importances) |\n| Validation and practical use | [Validation audit](work/notebooks/w06_validation_audit.ipynb) and [action playbook](work/notebooks/w07_action_playbook.ipynb) |\n\n### What the model is intended to support\n\nThe capstone compares Logistic Regression, Decision Tree and Random Forest models with a rule-based baseline. It uses a client-grouped holdout to evaluate ranking pages for human review. The label describes decline within an observed window; it does not establish future traffic recovery or the causal effect of refreshing content.\n\n**Evaluation status:** Random Forest Precision@50 is 68%, versus 42% for the baseline (26 percentage points higher). The seed-42 holdout contains 2,325 pages from 6 clients, with a declining-label base rate of 39.1%. Regenerate the notebook, metrics and reports with `python scripts/run_capstone.py` using the pinned capstone environment. These are exploratory current-state results with overlapping windows and full-snapshot preprocessing; see the report for limitations.\n\n## My work and the starter foundation\n\n- **My internship submissions:** the assignment notebooks and capstone report under [work/](work/), with saved summaries in [work/outputs/](work/outputs/).\n- **Supplied foundation:** the original [FlyRank starter template](https://github.com/flyrank-bih/flyrank-ml-internship-starter), training materials, anonymised sample, reference scripts and example outputs. These remain credited to FlyRank and the original contributors.\n- **Reference results:** the baseline/model comparison in the starter guide below belongs to the supplied reference pipeline. It is separate from the capstone evaluation above.\n\n**Stack:** Python · pandas · NumPy · scikit-learn · Jupyter/Colab · Matplotlib\n\nFor environment setup, see [SETUP.md](SETUP.md). For data permissions and handling, see [DATA_USE.md](DATA_USE.md).\n\n<details>\n<summary>Original starter guide, notebook launchers and reference pipeline</summary>\n\n## FlyRank ML Internship — Starter Repo\n\n**Applied Search Intelligence: Google Search Ranking & Discoverability**\n\nThis is the starting point for the FlyRank ML Internship. You **clone it into your own public\nrepo** (one click — *Use this template*), build everything there, and submit that repo URL on\neach assignment in your portal — it\'s your workspace, your submission, and your portfolio all\nat once. The rhythm is simple: do the work, commit it, submit on the card. Done.\n\nEverything here runs on a small **anonymized** slice of real FlyRank search data. No credentials,\nno private client data, no setup headaches.\n\n> **New here?** Two reads: **[SETUP.md](SETUP.md)** (GitHub, Colab, and data access — ten\n> minutes, with every silent pitfall flagged), then **[GUIDE.md](GUIDE.md)** (every file\n> explained, what to edit vs. leave alone, and where your own work goes — five minutes).\n\n---\n\n## Quickstart — first win in 2 minutes\n\nThe fastest path is Google Colab (one click, zero install). Open Notebook 1 and run all cells:\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)\n **Week 1 — Run it, then discover a real truth yourself**\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)\n **Week 2 — The model is just a rule you can read**\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)\n **Weeks 3+ — The full release (~79M rows) via DuckDB, no download needed** — hosted at\n [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse) (gated: request access + accept the data-use terms, approval is instant)\n\n---\n\n## Your assignment notebooks — open, fill, save, done\n\nEvery assignment is one pre-named skeleton notebook in `work/notebooks/`. Click its badge,\nfill the sections in order, then **File → Save a copy in GitHub → OK** — the dialog is\nalready pre-filled with your repo and the right path.\n\n> **The badges know whose repo they\'re in.** About 30 seconds after you create your copy, an\n> automatic commit ("Point Colab badges at this copy") rewires every badge in it to open\n> **your** notebooks — with your saved work — instead of the shared read-only ones. Reading\n> this on the shared starter page? The badges below open blank previews; make your copy\n> first ([SETUP.md](SETUP.md), Moment 1).\n\n| Week | Card | Notebook | Open |\n|---|---|---|---|\n| 1 | ML-02 | `w01_research_question` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true) |\n| 2 | ML-03 | `w02_ml_task_framing` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true) |\n| 3 | ML-04 | `w03_data_contract` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true) |\n| 3 | ML-05 | `w03_feature_leakage_check` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true) |\n| 4 | ML-06 | `w04_signal_audit` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true) |\n| 4 | ML-07 | `w04_baseline_score` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true) |\n| 5 | ML-08 | `w05_model` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true) |\n| 6 | ML-09 | `w06_validation_audit` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true) |\n| 7 | ML-10 | `w07_action_playbook` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true) |\n| 8 | ML-11 | `capstone` | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true) |\n\nBadges not opening *your* copy? Colab\'s built-in opener always works: **File → Open notebook\n→ GitHub tab** → paste `github.com/you/your-repo` → pick the notebook.\n\n### Prefer local?\n\n```bash\ngit clone https://github.com/Prajwal-Rd-bond/flyrank-ml-internship-starter.git\ncd flyrank-ml-internship-starter\npip install -r requirements.txt          # or: uv pip install -r requirements.txt\npython scripts/run_all.py\n```\n\nThat runs the whole pipeline on the bundled sample and writes results to `outputs/`.\n\n---\n\n## What you get\n\n| Path | What it is |\n|---|---|\n| `notebooks/` | Week 1–2 **first-win notebooks** (Colab-ready). Start here. |\n| `scripts/01–05` + `run_all.py` | The runnable reference pipeline: prepare → baseline → train → evaluate → PDF. |\n| `data/raw/content_refresh_anonymized.csv` | The anonymized starter dataset (~30k pages). |\n| `outputs/` | Example outputs so you can see the **target shape** (`model_report.md`, `refresh_queue_sample.csv`, `charts/`). |\n| `work/` | **Your space.** Lane experiments and your capstone live here — see `work/README.md`. |\n| `docs/` | The core docs + the data dictionary (see below). |\n\n### Read these (in `docs/`)\n\n1. **`ml-core-foundation-framework.md`** — the first-principles map of ML as a whole system. The backbone of the live sessions.\n2. **`ml-intern-dataset-and-lane-guide.md`** — how to use the data safely, the capstone workflow, and the analysis "lanes" you can pick from.\n3. **`intern-free-tooling-guide.md`** — the zero-budget tool stack (Python, Colab, free AI assistants). You never need to pay for anything.\n4. **`data-dictionary.md`** — all 44 columns: meaning, scale, and gotchas. Keep it open while you work.\n\n---\n\n## The pipeline (what `run_all.py` does)\n\n```text\n01_prepare_features.py   clean + build the feature vector, define the label\n02_baseline_score.py     a transparent hand-rule "fix this first" score\n03_train_model.py        logistic regression, decision tree, random forest (client-holdout split)\n04_evaluate_and_export.py  ranked queue + charts + Markdown report\n05_build_pdf_report.py   a shareable PDF summary\n```\n\nIn the supplied reference pipeline on the bundled sample, the learned model beats the hand-written rule at picking the right\npages to review first (**Precision@50 ≈ 0.24 → 0.74**; the model number can land 0.68–0.74\ndepending on library versions — the ~3x lift is the point). The notebooks compute these numbers\nlive, so they always reflect the current data and environment.\n\n**Teaching point:** the model is the capstone, but the *workflow* is the lesson —\n`problem framing → data cleaning → baseline → first model → evaluation → explainable recommendation`.\n\n---\n\n## Data safety (read `DATA_USE.md`)\n\n- Only the small **anonymized** CSV ships here — no client names, domains, URLs, titles, or keywords.\n- **Never** add raw private client data to this repo or your fork. Need more data? Request an approved\n  release from your mentor — never export it yourself.\n- Don\'t paste client data into third-party AI tools.\n- Frame every result as **observed / measured / directional / decision-support** — never\n  "I predicted Google\'s algorithm."\n\nThe `.gitignore` blocks datasets by default, and CI fails any commit that includes a dataset.\n\n---\n\n## Assignments & schedule\n\nWeekly assignments, live events, and the capstone live on **your portal board** (your\nenrollment email has your access link). This repo is the shared technical foundation they all\nbuild on — and the `skills/` folder here is the instruction library for your AI assistant\n(start at [skills/README.md](skills/README.md)).\n\n**First time with GitHub?** You need exactly four things (full walkthrough: [SETUP.md](SETUP.md)):\n1. A free account at github.com.\n2. Your own copy of this repo: **Use this template → Create a new repository** → public.\n   (One click — brings the notebooks, `work/`, and the CI leak-guard with it.)\n3. In Colab: *File → Save a copy in GitHub* — opened from your copy\'s badges, the dialog is\n   already pre-filled with your repo and path, so it\'s just OK (Colab handles auth).\n4. That\'s your submission repo — share its **github.com/you/your-repo** URL with Assignment 1\n   (never a colab.research.google.com or drive.google.com link).\n\n---\n\n*Track leads: Mirza Ašćerić (ML) · Hole (data engineering). Code under MIT (see `LICENSE`); data under `DATA_USE.md`.*\n\n\n</details>\n'}
for name, content in files.items():
    (root/name).write_text(content)
data = root/'data/raw/content_refresh_anonymized.csv'
if not data.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Prajwal-Rd-bond/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv', data)
assert hashlib.sha256(data.read_bytes()).hexdigest() == 'c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396', 'Dataset differs from local evaluation'
import importlib.metadata, platform
print('Python:', platform.python_version())
for package in ['numpy', 'pandas', 'scikit-learn', 'scipy']:
    print(package, importlib.metadata.version(package))
print('Public starter data verified against local SHA-256.')


# Model improvement experiment

Prajwal Ravi Devadiga · bounded training-only client-grouped selection. The historical holdout is not sealed. Results describe observed decline, not future outcomes.

In [ ]:
from pathlib import Path
import os, sys
while not Path("scripts/tune_capstone.py").exists() and Path.cwd()!=Path("/"):
    os.chdir("..")
sys.path.insert(0,str(Path.cwd()/"scripts"))
from tune_capstone import run
results=run()